In [1]:
import pandas as pd
import glob

reviews = pd.concat([pd.read_csv(f) for f in glob.glob("../data/reviews_*.csv")], ignore_index=True)
products = pd.read_csv("../data/product_info.csv")

C:\Users\andre\AppData\Local\Temp\ipykernel_43140\3924998639.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.concat([pd.read_csv(f) for f in glob.glob("../data/reviews_*.csv")], ignore_index=True)
C:\Users\andre\AppData\Local\Temp\ipykernel_43140\3924998639.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.concat([pd.read_csv(f) for f in glob.glob("../data/reviews_*.csv")], ignore_index=True)
C:\Users\andre\AppData\Local\Temp\ipykernel_43140\3924998639.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.concat([pd.read_csv(f) for f in glob.glob("../data/reviews_*.csv")], ignore_index=True)


In [2]:
# 1. Fusionner les deux datasets sur product_id
df = reviews.merge(products, on='product_id', suffixes=('_review', '_product'))

# 2. Supprimer les colonnes inutiles
cols_to_drop = [
    'Unnamed: 0', 'author_id', 'helpfulness', 'review_text', 'review_title',
    'submission_time', 'eye_color', 'hair_color', 'skin_tone',
    'value_price_usd', 'sale_price_usd', 'variation_desc',
    'brand_id', 'variation_type', 'variation_value',
    'child_count', 'child_max_price', 'child_min_price',
    'product_name_review', 'product_name_product', 'brand_name_review'
]
df = df.drop(columns=cols_to_drop)

# 3. Créer la variable cible
df['good_rating'] = (df['rating_review'] >= 4).astype(int)

# 4. Vérifier
print(df.shape)
print(df['good_rating'].value_counts())
print(df.isnull().sum())

(1094411, 25)
good_rating
1    898340
0    196071
Name: count, dtype: int64
rating_review                    0
is_recommended              167988
total_feedback_count             0
total_neg_feedback_count         0
total_pos_feedback_count         0
skin_type                   111557
product_id                       0
price_usd_review                 0
brand_name_product               0
loves_count                      0
rating_product                   0
reviews                          0
size                         43363
ingredients                  22025
price_usd_product                0
limited_edition                  0
new                              0
online_only                      0
out_of_stock                     0
sephora_exclusive                0
highlights                  113936
primary_category                 0
secondary_category               0
tertiary_category           161256
good_rating                      0
dtype: int64


In [3]:
# Supprimer les lignes sans skin_type (trop de NaN pour imputer)
df = df.dropna(subset=['skin_type'])

# Remplir les manquants restants
df['size'] = df['size'].fillna('Unknown')
df['ingredients'] = df['ingredients'].fillna('Unknown')
df['highlights'] = df['highlights'].fillna('Unknown')
df['tertiary_category'] = df['tertiary_category'].fillna('Unknown')
df['is_recommended'] = df['is_recommended'].fillna(df['is_recommended'].median())

# Vérifier
print(df.shape)
print(df.isnull().sum().sum(), "valeurs manquantes restantes")

(982854, 25)
0 valeurs manquantes restantes


## Phase 3 : Préparation des features 

In [4]:
# Corrélation des variables numériques avec good_rating
numeric_cols = [
    'price_usd_review', 'is_recommended', 'total_feedback_count',
    'total_pos_feedback_count', 'total_neg_feedback_count',
    'loves_count', 'reviews', 'limited_edition', 'new',
    'online_only', 'out_of_stock', 'sephora_exclusive'
]

correlations = df[numeric_cols + ['good_rating']].corr()['good_rating'].drop('good_rating').sort_values(ascending=False)
print(correlations)

is_recommended              0.839400
new                         0.030494
price_usd_review            0.015805
online_only                 0.010861
reviews                     0.006701
limited_edition            -0.000775
out_of_stock               -0.004503
sephora_exclusive          -0.012959
loves_count                -0.015287
total_pos_feedback_count   -0.053758
total_feedback_count       -0.083338
total_neg_feedback_count   -0.156975
Name: good_rating, dtype: float64


In [5]:
from sklearn.preprocessing import LabelEncoder

features = [
    'is_recommended',
    'total_neg_feedback_count',
    'total_feedback_count', 
    'total_pos_feedback_count',
    'price_usd_review',
    'loves_count',
    'reviews',
    'new',
    'online_only',
    'skin_type',
    'primary_category',
    'secondary_category'
]

df_model = df[features + ['good_rating']].copy()

# Encoder les variables catégorielles
le = LabelEncoder()
for col in ['skin_type', 'primary_category', 'secondary_category']:
    df_model[col] = le.fit_transform(df_model[col])

print(df_model.shape)
print(df_model.dtypes)

(982854, 13)
is_recommended              float64
total_neg_feedback_count      int64
total_feedback_count          int64
total_pos_feedback_count      int64
price_usd_review            float64
loves_count                   int64
reviews                     float64
new                           int64
online_only                   int64
skin_type                     int64
primary_category              int64
secondary_category            int64
good_rating                   int64
dtype: object


In [6]:
from sklearn.model_selection import train_test_split

# Séparer features et cible
X = df_model.drop(columns=['good_rating'])
y = df_model['good_rating']

# Split 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Sauvegarder le dataset propre
df_model.to_csv('../data/processed_dataset.csv', index=False)

print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.value_counts())

X_train : (786283, 12)
X_test : (196571, 12)
y_train : good_rating
1    644854
0    141429
Name: count, dtype: int64
